# ETL source: electricity Gold

Add to the SAME pipeline as `02_silver_electricity_etl.ipynb`.
Output: `db_labb2.etl_demo.gold_annual_electricity_price` (materialized view).

Keeps the original Gold electricity summary exactly: complete 2025 only,
SEK/kWh weighted by each interval's duration, 4 areas, 525600 minutes and
15387 intervals per area. Partial 2026 is intentionally excluded.
All 2026 intervals remain available in the demo Silver view.
If using a small sample instead of a complete 2025 Bronze history, run Silver
alone first; this Gold definition deliberately rejects an incomplete year.


In [ ]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F


@dp.materialized_view(
    name="gold_annual_electricity_price",
    comment="Complete-year 2025 time-weighted electricity prices for SE1-SE4",
)
@dp.expect_or_fail("complete_2025", "year = 2025 AND total_minutes = 525600 AND intervals = 15387")
@dp.expect_or_fail("annual_price_present", "avg_sek_per_kwh IS NOT NULL")
def gold_annual_electricity_price():
    prices = spark.read.table("silver_electricity_prices")
    annual = (
        prices.where(F.year("price_date") == 2025)
        .withColumn("year", F.year("price_date"))
        .groupBy("price_area", "year")
        .agg(
            (F.sum(F.col("sek_per_kwh") * F.col("duration_minutes")) /
             F.sum("duration_minutes")).alias("avg_sek_per_kwh"),
            F.sum("duration_minutes").alias("total_minutes"),
            F.count(F.lit(1)).alias("intervals"),
        )
    )
    # An explicit four-area grid makes absent 2025 data FAIL rather than
    # returning an empty result that passes row-level expectations vacuously.
    expected = spark.range(4).select(
        F.concat(F.lit("SE"), (F.col("id")+1).cast("string")).alias("price_area"),
        F.lit(2025).alias("year"),
    )
    return expected.join(annual, ["price_area", "year"], "left")
